# Perfil da turma: linguagens e formatura

## Objetivo

Criar visualizacoes estaticas para observar o perfil da turma em relacao ao contato com linguagens de programacao e ao tempo restante para a formatura.

## Pergunta analisada

Como a quantidade de linguagens conhecidas aparece na turma e como ela se distribui entre alunos mais perto ou mais longe da formatura?

## Descricao dos dados

O arquivo `dataset-turma.csv` contem respostas da turma sobre conectividade, tempo restante para formatura, linguagens ja usadas, disciplinas cursadas e outras informacoes de perfil. Neste notebook, usei principalmente as colunas de semestres restantes e linguagens de programacao.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.plotting.register_matplotlib_converters()
%matplotlib inline


## Carregamento dos dados

O dataset e carregado da pasta `data` do repositorio, sem upload manual pelo Google Colab.


In [ ]:
turma_filepath = Path("../data/dataset-turma.csv")

turma_data = pd.read_csv(turma_filepath, sep=";")

turma_data.head()


## Transformacao feita

Selecionei as colunas necessarias para as duas visualizacoes. A coluna de semestres foi convertida para numero, e a coluna de linguagens foi transformada em uma contagem de linguagens marcadas por aluno.


In [ ]:
# colunas
col_semestres = "Quantos semestres faltam para você se formar?"
col_linguagens = "Marque as linguagens de programação que você já teve algum contato prático:"

dados = turma_data[[col_semestres, col_linguagens]].copy()

# numero de semestres
dados["semestres_restantes"] = (
    dados[col_semestres]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(int)
)

# quantidade de linguagens marcadas por cada aluno
dados["quantidade_linguagens"] = (
    dados[col_linguagens]
    .fillna("")
    .apply(lambda linguagens: len([ling.strip() for ling in linguagens.split(",") if ling.strip()]))
)

dados[["semestres_restantes", "quantidade_linguagens"]].head()

## Projeto 1: quantidade de linguagens conhecidas

### Justificativa da visualizacao

Usei um grafico de barras porque a pergunta compara quantos alunos aparecem em cada quantidade de linguagens conhecidas. O destaque nas barras mais frequentes ajuda a mostrar rapidamente onde a turma se concentra.


In [ ]:
# Quantidade de alunos
n_alunos = len(dados)

# Conta quantos alunos aparecem em cada quantidade de linguagens
contagem_linguagens = (
    dados["quantidade_linguagens"]
    .value_counts()
    .sort_index()
    .reset_index()
)

# Renomeia as colunas do grafico
contagem_linguagens.columns = ["quantidade_linguagens", "quantidade_alunos"]

In [ ]:
# Tamanho da figura
fig, ax = plt.subplots(figsize=(15, 8))

# Total de alunos considerados no gráfico
n_alunos = len(dados)

# Cores utilizadas
cor_base = "#C7D1DE"
cor_destaque = "#159A8C"
cor_texto = "#1F2A3D"
cor_secundaria = "#4A5870"

# Paleta com destaque para as quantidades mais representativas
cores = {
    qtd: cor_destaque if qtd in [3, 4] else cor_base
    for qtd in contagem_linguagens["quantidade_linguagens"]
}

# Gráfico de barras mostrando a quantidade de alunos por quantidade de linguagens
sns.barplot(
    data=contagem_linguagens,
    x="quantidade_linguagens",
    y="quantidade_alunos",
    hue="quantidade_linguagens",
    palette=cores,
    dodge=False,
    legend=False,
    ax=ax
)

# Adiciona a quantidade de alunos acima das barras
for i, linha in enumerate(contagem_linguagens.itertuples()):
    qtd_alunos = int(linha.quantidade_alunos)

    ax.text(
        i,
        qtd_alunos + 0.45,
        f"{qtd_alunos}",
        ha="center",
        va="bottom",
        fontsize=20,
        fontweight="bold",
        color="black"
    )

# Título
fig.suptitle(
    "Alunos da turma versus quantidade de linguagens de programação conhecida",
    fontsize=24,
    fontweight="bold",
    color=cor_texto,
    y=0.98
)

# Informação geral do gráfico
ax.text(
    0.98,
    0.93,
    f"Total: {n_alunos} alunos",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=14,
    fontweight="bold",
    color=cor_secundaria,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#E6ECF3",
        linewidth=1
    )
)

# Ajustes dos eixos
ax.set_xlabel(
    "Quantidade de Linguagens Conhecidas",
    fontsize=16,
    fontweight="bold",
    color=cor_texto,
    labelpad=18
)

ax.set_ylabel("")

ax.set_xticks(range(len(contagem_linguagens)))
ax.set_xticklabels(
    [f"{int(qtd)} Lgs." for qtd in contagem_linguagens["quantidade_linguagens"]],
    fontsize=15,
    color=cor_texto
)

ax.set_yticks([])
ax.set_ylim(0, contagem_linguagens["quantidade_alunos"].max() + 5)

# Remove elementos que não ajudam diretamente na leitura
sns.despine(ax=ax, left=True, right=True, top=True)

ax.spines["bottom"].set_color("#DCE3EC")
ax.spines["bottom"].set_linewidth(1.2)

ax.tick_params(axis="x", length=0, pad=8)

# Fonte dos dados
fig.text(
    0.5,
    0.02,
    "Fonte: dataset-turma.csv",
    ha="center",
    fontsize=13,
    fontweight="medium",
    color="#5F6F85",
    style="italic"
)

plt.tight_layout(rect=[0, 0.05, 1, 0.92])
plt.savefig("lab2-projeto1.png", dpi=300, bbox_inches="tight")
plt.show()

## Projeto 2: linguagens e tempo para formatura

### Justificativa da visualizacao

Usei boxplot porque a pergunta envolve comparar distribuicoes entre grupos de semestres restantes. Assim, e possivel ver mediana, variacao e valores mais afastados em cada grupo.


In [ ]:
# Cria rotulos mais legíveis para os semestres restantes
rotulos_semestres = {
    4: "4 semestres",
    3: "3 semestres",
    2: "2 semestres",
    1: "1 semestre",
    0: "0 (último)"
}

ordem_semestres = ["4 semestres", "3 semestres", "2 semestres", "1 semestre", "0 (último)"]

dados["tempo_formatura"] = dados["semestres_restantes"].map(rotulos_semestres)

# Estatisticas resumidas por grupo
resumo_semestres = (
    dados
    .groupby("tempo_formatura")["quantidade_linguagens"]
    .agg(["count", "min", "median", "mean", "max"])
    .reindex(ordem_semestres)
)

resumo_semestres

In [ ]:
# Mediana geral da turma
mediana_turma = dados["quantidade_linguagens"].median()
mediana_turma

In [ ]:
# tamanho da figura
fig, ax = plt.subplots(figsize=(15, 8))

# Quantidade de alunos no dataset
n_alunos = len(dados)

mediana_turma = dados["quantidade_linguagens"].median()

ordem_semestres_dados = [
    "0 (último)",
    "1 semestre",
    "2 semestres",
    "3 semestres",
    "4 semestres"
]

# eixo x
rotulos_semestres = [
    "0 semestre",
    "1 semestre",
    "2 semestres",
    "3 semestres",
    "4 semestres"
]

# Cores utilizadas
cor_box = "#DDECEA"
cor_borda = "#159A8C"
cor_mediana = "#D95F73"
cor_texto = "#1F2A3D"
cor_secundaria = "#4A5870"
cor_grade = "#DDE6F0"

# Boxplot mostrando a distribuição de linguagens por tempo para formatura
sns.boxplot(
    data=dados,
    x="tempo_formatura",
    y="quantidade_linguagens",
    order=ordem_semestres_dados,
    width=0.48,
    color=cor_box,
    linewidth=1.8,
    boxprops={
        "edgecolor": cor_borda,
        "facecolor": cor_box,
        "alpha": 0.90
    },
    whiskerprops={
        "color": cor_borda,
        "linewidth": 1.8
    },
    capprops={
        "color": cor_borda,
        "linewidth": 1.8
    },
    medianprops={
        "color": cor_borda,
        "linewidth": 2.3
    },
    flierprops={
        "marker": "o",
        "markerfacecolor": cor_borda,
        "markeredgecolor": cor_borda,
        "markersize": 5,
        "alpha": 0.65
    },
    ax=ax
)

# Linha da mediana geral da turma
ax.axhline(
    mediana_turma,
    color=cor_mediana,
    linestyle="--",
    linewidth=2,
    label=f"Mediana da turma ({mediana_turma:.0f} Lgs.)"
)

# Título
fig.suptitle(
    "Relação entre tempo para formatura e quantidade de linguagens conhecidas",
    fontsize=24,
    fontweight="bold",
    color=cor_texto,
    y=0.97
)

# Ajustes dos eixos
ax.set_xlabel(
    "Tempo Restante para a Formatura",
    fontsize=15,
    fontweight="bold",
    color=cor_texto,
    labelpad=18
)

ax.set_ylabel(
    "Quantidade de Linguagens Conhecidas",
    fontsize=15,
    fontweight="bold",
    color=cor_texto,
    labelpad=18
)

ax.set_ylim(1.5, 8.5)

ax.set_yticks(range(2, 9))
ax.set_yticklabels(
    [f"{valor} Lgs." for valor in range(2, 9)],
    fontsize=13,
    color=cor_secundaria
)

ax.set_xticks(range(len(ordem_semestres_dados)))
ax.set_xticklabels(
    rotulos_semestres,
    fontsize=14,
    color=cor_texto
)

# eixo y
ax.grid(
    axis="y",
    linestyle=":",
    linewidth=1,
    color=cor_grade
)

ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)

sns.despine(ax=ax, right=True, top=True)

ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#DCE3EC")
ax.spines["bottom"].set_linewidth(1.2)

ax.tick_params(axis="both", length=0)

# Legenda
ax.legend(
    loc="upper right",
    frameon=True,
    facecolor="white",
    edgecolor="#E6ECF3",
    fontsize=12
)

# Fonte dos dados
fig.text(
    0.5,
    0.02,
    f"Fonte: dataset-turma.csv",
    ha="center",
    fontsize=13,
    fontweight="medium",
    color="#5F6F85",
    style="italic"
)

plt.tight_layout(rect=[0, 0.05, 1, 0.92])
plt.savefig("lab2-projeto2.png", dpi=300, bbox_inches="tight")
plt.show()

## Conclusao

No primeiro grafico, a maior parte da turma se concentra em algumas quantidades de linguagens, principalmente nas faixas destacadas. No segundo grafico, da para comparar se os alunos mais perto da formatura apresentam uma distribuicao diferente de linguagens conhecidas em relacao aos demais grupos. A leitura geral ajuda a descrever o perfil tecnico da turma sem assumir uma relacao causal entre semestre e experiencia.
